# Hunyuan3D-2.0 Turbo 테스트 (T4 GPU)
단일 이미지 → 3D 모델 변환

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 저장소 클론 및 패키지 설치
!git clone https://github.com/Tencent/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# 3. custom rasterizer 빌드 (텍스처용)
!pip install ninja -q
%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!python setup.py install -q
%cd /content/Hunyuan3D-2

In [ ]:
# 4. 이미지 업로드
from google.colab import files
uploaded = files.upload()
import os
image_path = list(uploaded.keys())[0]
print(f'업로드된 이미지: {image_path}')

In [ ]:
# 6. 모델 다운로드 및 3D 생성 (메모리 최적화)
import sys, os, gc
import torch

repo_path = '/content/Hunyuan3D-2'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)
os.makedirs('/content/output', exist_ok=True)

# rembg 모델 메모리에서 제거
gc.collect()
torch.cuda.empty_cache()

from huggingface_hub import snapshot_download

cache_path = '/root/.cache/hy3dgen/tencent/Hunyuan3D-2'
print('turbo 모델 다운로드 중...')
snapshot_download(
    repo_id='tencent/Hunyuan3D-2',
    local_dir=cache_path,
    allow_patterns=['hunyuan3d-dit-v2-0-turbo/**'],
)
print('다운로드 완료!')

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from PIL import Image

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2',
    subfolder='hunyuan3d-dit-v2-0-turbo',
    use_safetensors=True,
)
pipeline = pipeline.to('cuda')

# 추론 전 메모리 최대한 비우기
gc.collect()
torch.cuda.empty_cache()

ram_before = os.popen('free -m').read()
print('추론 전 메모리:\n', ram_before)

image = Image.open(bg_removed_path)
print('3D 생성 중...')

# no_grad + autocast으로 메모리 절약
with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
    mesh = pipeline(image=image, num_inference_steps=5)[0]

mesh.export('/content/output/result.glb')

# 파이프라인 메모리 해제
del pipeline
gc.collect()
torch.cuda.empty_cache()
print('완료! VRAM 해제됨')

In [ ]:
# 6. 모델 다운로드 및 3D 생성
import sys, os

repo_path = '/content/Hunyuan3D-2'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)
os.makedirs('/content/output', exist_ok=True)

from huggingface_hub import snapshot_download

local_model_path = '/root/.cache/hunyuan3d/tencent/Hunyuan3D-2'
print('turbo 모델 다운로드 중... (~800MB, 수 분 소요)')
snapshot_download(
    repo_id='tencent/Hunyuan3D-2',
    local_dir=local_model_path,
    allow_patterns=['hunyuan3d-dit-v2-0-turbo/**'],
)
print('다운로드 완료!')

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from PIL import Image

subfolder_path = f'{local_model_path}/hunyuan3d-dit-v2-0-turbo'
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    subfolder_path,
    use_safetensors=True,
)
pipeline = pipeline.to('cuda')

image = Image.open(bg_removed_path)
print('3D 생성 중...')
mesh = pipeline(image=image, num_inference_steps=5)[0]
mesh.export('/content/output/result.glb')
print('완료!')

In [ ]:
# 7. (선택) 텍스처 입히기
import torch
torch.cuda.empty_cache()

from huggingface_hub import snapshot_download
print('paint 모델 다운로드 중...')
snapshot_download(
    repo_id='tencent/Hunyuan3D-2',
    local_dir=local_model_path,
    allow_patterns=['hunyuan3d-paint-v2-0-turbo/**'],
)

from hy3dgen.texgen import Hunyuan3DPaintPipeline
import trimesh

paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained(
    f'{local_model_path}/hunyuan3d-paint-v2-0-turbo'
)
paint_pipeline = paint_pipeline.to('cuda')

mesh = trimesh.load('/content/output/result.glb')
image = Image.open(bg_removed_path)
textured = paint_pipeline(mesh=mesh, image=image)
textured.export('/content/output/result_textured.glb')
print('텍스처 완료!')

In [ ]:
# 8. 결과 다운로드
import glob
from google.colab import files

output_files = glob.glob('/content/output/*.*')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'{f}  ({size_mb:.1f} MB)')
    files.download(f)